# 机械臂扭矩曲线预测 - 训练Notebook

本notebook用于训练深度学习模型来预测机械臂拧盖子的力学曲线

**任务说明**:
- 输入: 前N个时间步的数据 (例如: 0-100, 即前1秒)
- 输出: 后M个时间步的signal_1预测 (例如: 101-500, 即后4秒)
- 数据: Time(s), Torque, signal_0, signal_1, signal_2
- 优先训练signal_1，效果好再扩展到signal_0和signal_2

## 1. 环境设置和依赖安装

In [ ]:
# 检查GPU
!nvidia-smi

In [ ]:
# 克隆代码仓库（如果需要）
# !git clone https://github.com/your-repo/Super-Strawberry.git
# %cd Super-Strawberry

In [ ]:
# 安装依赖
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install pandas numpy matplotlib seaborn scikit-learn tqdm

## 2. 导入库和模块

In [ ]:
import sys
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 添加src目录到路径
sys.path.append('./src')

# 导入自定义模块
from data_loader import load_data, get_sample_data_info
from model import get_model
from train import Trainer, predict_batch, calculate_metrics
from evaluate import evaluate_model, plot_training_history, plot_predictions

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 3. 配置参数

In [ ]:
# ==================== 数据参数 ====================
DATA_DIR = './data'  # CSV文件目录
CSV_PATTERN = '*_open.csv'  # CSV文件匹配模式

# ==================== 序列参数 ====================
INPUT_LENGTH = 100   # 输入序列长度 (前100个时间步 = 1秒)
OUTPUT_LENGTH = 400  # 输出序列长度 (后400个时间步 = 4秒)

# ==================== 训练参数 ====================
SIGNAL_TYPE = 'signal_1'  # 要预测的信号 ('signal_0', 'signal_1', 'signal_2')
USE_ALL_FEATURES = True   # 是否使用所有特征作为输入
TRAIN_SPLIT = 0.8         # 训练集比例
BATCH_SIZE = 32           # 批次大小
LEARNING_RATE = 0.001     # 学习率
EPOCHS = 100              # 训练轮数
EARLY_STOPPING_PATIENCE = 15  # 早停耐心值

# ==================== 模型参数 ====================
MODEL_TYPE = 'simple_lstm'  # 模型类型: 'lstm', 'gru', 'transformer', 'simple_lstm'
HIDDEN_DIM = 128            # 隐藏层维度
NUM_LAYERS = 3              # 网络层数
DROPOUT = 0.2               # Dropout比例

# ==================== 设备设置 ====================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ==================== 保存路径 ====================
MODEL_SAVE_DIR = './models'
RESULTS_DIR = './results'

print("Configuration:")
print(f"  Data Directory: {DATA_DIR}")
print(f"  Input Length: {INPUT_LENGTH}")
print(f"  Output Length: {OUTPUT_LENGTH}")
print(f"  Signal Type: {SIGNAL_TYPE}")
print(f"  Model Type: {MODEL_TYPE}")
print(f"  Device: {DEVICE}")

## 4. 数据探索（可选）

In [ ]:
# 查看数据文件
import glob
csv_files = glob.glob(os.path.join(DATA_DIR, CSV_PATTERN))
print(f"Found {len(csv_files)} CSV files")
if csv_files:
    print("\nFirst 5 files:")
    for f in csv_files[:5]:
        print(f"  {f}")

In [ ]:
# 查看单个CSV文件的详细信息
if csv_files:
    sample_df = get_sample_data_info(csv_files[0])

In [ ]:
# 可视化样本数据
if csv_files:
    df = pd.read_csv(csv_files[0])
    
    fig, axes = plt.subplots(3, 1, figsize=(15, 10))
    
    axes[0].plot(df['Time(s)'], df['signal_0'], label='signal_0', linewidth=1.5)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Signal Value')
    axes[0].set_title('Signal 0')
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    
    axes[1].plot(df['Time(s)'], df['signal_1'], label='signal_1', color='orange', linewidth=1.5)
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Signal Value')
    axes[1].set_title('Signal 1 (Target)')
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    
    axes[2].plot(df['Time(s)'], df['signal_2'], label='signal_2', color='green', linewidth=1.5)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Signal Value')
    axes[2].set_title('Signal 2')
    axes[2].grid(True, alpha=0.3)
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()

## 5. 加载数据

In [ ]:
# 加载训练集和测试集
train_loader, test_loader = load_data(
    data_dir=DATA_DIR,
    pattern=CSV_PATTERN,
    train_split=TRAIN_SPLIT,
    input_length=INPUT_LENGTH,
    output_length=OUTPUT_LENGTH,
    signal_type=SIGNAL_TYPE,
    use_all_features=USE_ALL_FEATURES,
    batch_size=BATCH_SIZE
)

# 查看一个批次的数据
for inputs, targets in train_loader:
    print(f"Input shape: {inputs.shape}")   # [batch_size, input_length, features]
    print(f"Target shape: {targets.shape}") # [batch_size, output_length]
    break

## 6. 创建模型

In [ ]:
# 确定输入维度
if USE_ALL_FEATURES:
    input_dim = 5  # Time, Torque, signal_0, signal_1, signal_2
else:
    input_dim = 1  # 仅使用目标信号

# 创建模型
model = get_model(
    model_type=MODEL_TYPE,
    input_dim=input_dim,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    output_length=OUTPUT_LENGTH,
    dropout=DROPOUT
)

# 打印模型结构
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 7. 训练模型

In [ ]:
# 创建训练器
trainer = Trainer(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    device=DEVICE,
    learning_rate=LEARNING_RATE,
    save_dir=MODEL_SAVE_DIR
)

# 开始训练
history = trainer.train(
    epochs=EPOCHS,
    early_stopping_patience=EARLY_STOPPING_PATIENCE
)

## 8. 可视化训练历史

In [ ]:
# 绘制训练历史
plot_training_history(
    history,
    save_path=os.path.join(RESULTS_DIR, 'training_history.png')
)

## 9. 加载最佳模型并评估

In [ ]:
# 加载最佳模型
trainer.load_checkpoint('best_model.pth')
print("Best model loaded!")

In [ ]:
# 完整评估
metrics, predictions, targets, inputs = evaluate_model(
    model=trainer.model,
    data_loader=test_loader,
    device=DEVICE,
    save_dir=RESULTS_DIR
)

## 10. 单样本预测测试

In [ ]:
# 选择一个测试样本进行预测
from evaluate import predict_single_sequence

# 获取一个测试样本
test_input, test_target = next(iter(test_loader))
sample_input = test_input[0]  # 第一个样本
sample_target = test_target[0].numpy()

# 预测
sample_prediction = predict_single_sequence(
    trainer.model,
    sample_input,
    device=DEVICE
)

# 可视化单个预测
plt.figure(figsize=(15, 5))

input_time = np.arange(0, INPUT_LENGTH)
output_time = np.arange(INPUT_LENGTH, INPUT_LENGTH + OUTPUT_LENGTH)

# 绘制输入
if USE_ALL_FEATURES:
    plt.plot(input_time, sample_input[:, -2].numpy(), 'b-', 
             label='Input Sequence', linewidth=2)
else:
    plt.plot(input_time, sample_input[:, 0].numpy(), 'b-', 
             label='Input Sequence', linewidth=2)

# 绘制真实值和预测值
plt.plot(output_time, sample_target, 'g-', label='Ground Truth', linewidth=2)
plt.plot(output_time, sample_prediction, 'r--', label='Prediction', linewidth=2)
plt.axvline(x=INPUT_LENGTH, color='gray', linestyle='--', alpha=0.5)

plt.xlabel('Time Step', fontsize=12)
plt.ylabel('Signal Value', fontsize=12)
plt.title('Single Sample Prediction', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 计算误差
mse = np.mean((sample_prediction - sample_target) ** 2)
mae = np.mean(np.abs(sample_prediction - sample_target))
print(f"\nSingle Sample Metrics:")
print(f"MSE: {mse:.6f}")
print(f"MAE: {mae:.6f}")

## 11. 保存最终结果

In [ ]:
# 保存配置信息
config = {
    'model_type': MODEL_TYPE,
    'input_length': INPUT_LENGTH,
    'output_length': OUTPUT_LENGTH,
    'signal_type': SIGNAL_TYPE,
    'use_all_features': USE_ALL_FEATURES,
    'hidden_dim': HIDDEN_DIM,
    'num_layers': NUM_LAYERS,
    'dropout': DROPOUT,
    'batch_size': BATCH_SIZE,
    'learning_rate': LEARNING_RATE,
    'metrics': metrics
}

import json
with open(os.path.join(RESULTS_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=4)

print(f"Configuration saved to {os.path.join(RESULTS_DIR, 'config.json')}")

## 12. 下载模型和结果（可选）

In [ ]:
# 压缩模型和结果文件
!zip -r results.zip models/ results/

# 在Colab中下载
from google.colab import files
files.download('results.zip')

## 13. 后续步骤

如果signal_1的预测效果好，可以继续训练signal_0和signal_2:

1. 修改配置中的`SIGNAL_TYPE`为`'signal_0'`或`'signal_2'`
2. 重新运行训练流程
3. 比较不同信号的预测效果

**模型改进建议**:
- 尝试不同的模型类型 (LSTM, GRU, Transformer)
- 调整超参数 (hidden_dim, num_layers, dropout)
- 尝试不同的输入/输出长度比例
- 添加注意力机制
- 尝试多任务学习（同时预测3个信号）